# Выгрузка PortWatch: Суэцкий канал, 2020–2026

Ноутбук получает ежедневные данные из официального IMF PortWatch API, фильтрует Суэцкий канал (`portid = chokepoint1`) и сохраняет результат в CSV, Excel (`.xlsx`) и Parquet.

**Для Excel открывайте файл `.xlsx`** — столбцы будут разделены корректно. CSV дополнительно сохраняется с разделителем `;`, который обычно подходит для русской локали Excel.

**Важно:** данные показывают число проходов судов и оценочную грузовместимость, а не точный тоннаж именно карбамида. Источник: [IMF PortWatch](https://portwatch.imf.org/).

In [ ]:
%pip install -q requests pandas pyarrow matplotlib openpyxl

In [ ]:
from pathlib import Path
import requests
import pandas as pd
import matplotlib.pyplot as plt

API_URL = (
    "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/"
    "Daily_Chokepoints_Data/FeatureServer/0/query"
)
START_YEAR = 2020
END_DATE = pd.Timestamp("2026-10-18")
PAGE_SIZE = 1000  # лимит API на одну страницу

FIELDS = [
    "date", "year", "month", "day", "portid", "portname",
    "n_container", "n_dry_bulk", "n_general_cargo", "n_roro",
    "n_tanker", "n_cargo", "n_total",
    "capacity_container", "capacity_dry_bulk", "capacity_general_cargo",
    "capacity_roro", "capacity_tanker", "capacity_cargo", "capacity",
    "ObjectId",
]

all_features = []
offset = 0
previous_first_id = None

while True:
    params = {
        "where": f"portid = 'chokepoint1' AND year >= {START_YEAR}",
        "outFields": ",".join(FIELDS),
        "returnGeometry": "false",
        "orderByFields": "date ASC",
        "resultRecordCount": PAGE_SIZE,
        "resultOffset": offset,
        "f": "json",
    }
    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(f"PortWatch API вернул ошибку: {payload['error']}")

    features = payload.get("features", [])
    if not features:
        break

    first_id = features[0].get("attributes", {}).get("ObjectId")
    if offset > 0 and first_id == previous_first_id:
        raise RuntimeError(
            "API повторил ту же страницу. Остановил выгрузку, чтобы не сохранить дубликаты. "
            "Попробуйте запустить ячейку ещё раз или уменьшить PAGE_SIZE."
        )
    previous_first_id = first_id
    all_features.extend(feature.get("attributes", {}) for feature in features)
    print(f"Получено записей: {len(all_features)}")

    if len(features) < PAGE_SIZE:
        break
    offset += len(features)

if not all_features:
    raise RuntimeError("API не вернул записи. Проверьте доступ к интернету и статус PortWatch.")

df = pd.DataFrame(all_features)

# ArcGIS может отдавать дату как миллисекунды Unix или как текст — обработаем оба варианта.
if pd.api.types.is_numeric_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
else:
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize()

df = df[(df["date"] >= pd.Timestamp(f"{START_YEAR}-01-01")) & (df["date"] <= END_DATE)].copy()
df = df.sort_values("date").drop_duplicates(subset=["date", "portid"])
df = df.reset_index(drop=True)

print(f"\nПериод фактических данных: {df['date'].min().date()} — {df['date'].max().date()}")
print(f"Количество строк: {len(df):,}")
print(f"Количество колонок: {len(df.columns)}")
display(df.head())

## Сохранение файлов

CSV удобно открыть в Excel, Parquet лучше подходит для дальнейшего анализа в Python и автоматического пайплайна.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

csv_path = Path("suez_portwatch_2020_2026_excel.csv")
xlsx_path = Path("suez_portwatch_2020_2026.xlsx")
parquet_path = Path("suez_portwatch_2020_2026.parquet")

# CSV with semicolon delimiter: opens in Russian-locale Excel as separate columns.
df.to_csv(csv_path, index=False, sep=";", encoding="utf-8-sig", decimal=",")

# XLSX is the most reliable option for opening directly in Excel.
with pd.ExcelWriter(xlsx_path, engine="openpyxl", datetime_format="yyyy-mm-dd") as writer:
    df.to_excel(writer, index=False, sheet_name="Suez PortWatch")
    ws = writer.sheets["Suez PortWatch"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 30
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(vertical="center", wrap_text=True)
    for col_idx, column_cells in enumerate(ws.columns, start=1):
        values = [str(c.value) if c.value is not None else "" for c in list(column_cells)[:200]]
        width = min(max(max((len(v) for v in values), default=10) + 2, 12), 28)
        ws.column_dimensions[get_column_letter(col_idx)].width = width
    # Format date column for easy filtering and reading.
    headers = {cell.value: cell.column for cell in ws[1]}
    if "date" in headers:
        for cell in ws.iter_cols(min_col=headers["date"], max_col=headers["date"], min_row=2, max_row=ws.max_row):
            for item in cell:
                item.number_format = "yyyy-mm-dd"

df.to_parquet(parquet_path, index=False)

print(f"Excel (рекомендуется): {xlsx_path.resolve()} ({xlsx_path.stat().st_size:,} байт)")
print(f"CSV для русской версии Excel: {csv_path.resolve()} ({csv_path.stat().st_size:,} байт)")
print(f"Parquet для Python: {parquet_path.resolve()} ({parquet_path.stat().st_size:,} байт)")

## Быстрая проверка данных

Графики ниже помогут увидеть перебои в движении судов. `capacity` — оценка суммарной грузовместимости, а не прямой объём перевозимого карбамида.

In [ ]:
daily = df.set_index("date").sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
daily["n_total"].plot(ax=ax)
ax.set_title("Суэцкий канал: число проходов судов в день")
ax.set_xlabel("Дата")
ax.set_ylabel("Число судов")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
daily["capacity"].plot(ax=ax)
ax.set_title("Суэцкий канал: оценочная грузовместимость проходящих судов")
ax.set_xlabel("Дата")
ax.set_ylabel("Оценка грузовместимости (единицы PortWatch)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(df[["date", "n_total", "n_container", "n_dry_bulk", "n_tanker", "capacity"]].tail(10))

## Примечания для проекта

- Пропуски и необычные провалы в AIS-данных нужно проверять отдельно: они могут отражать как реальные нарушения судоходства, так и ограничения наблюдения.
- Для сравнения с ценой карбамида месячные цены World Bank лучше сопоставить с месячными агрегатами PortWatch (например, средним или суммой проходов за месяц).
- В отчёте укажите источник: **IMF PortWatch, Daily Chokepoints Data**.

In [ ]:
# Если ноутбук запущен в Google Colab, автоматически скачать файлы.
try:
    from google.colab import files
    files.download(str(xlsx_path))
    files.download(str(csv_path))
    files.download(str(parquet_path))
except ImportError:
    print("Файлы сохранены в папке ноутбука. Для Excel откройте .xlsx; для Python используйте .parquet.")